# SRILM n-gram language models on the Brown corpus

In-class activity for the Foundation of Vision & Language course at IIT Jodhpur.

The notebook runs in Google Colab and calls the SRILM binaries (`ngram-count`, `ngram`) from Google Drive. See the README for the folder layout it expects.

In [ ]:
# Mount Google Drive so the SRILM binaries and the Brown data are reachable
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Move into the activity folder on Drive (change this path if yours is different)
# and make the two SRILM binaries executable
import os
os.chdir('/content/drive/My Drive/SRILM-LAB/InClassActivity2')
!chmod +x ngram ngram-count

In [ ]:
# Shared setup used by every exercise below
import re
import subprocess
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Brown corpus splits that come with the activity
train_file = 'browndata/brown-train.txt'
dev_file = 'browndata/brown-dev.txt'
test_file = 'browndata/brown-test.txt'

# Run a shell command and return its output as text
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout + r.stderr

# Train an n-gram model of the given order and smoothing options
def train(name, order, smoothing, text=train_file):
    sh(f'./ngram-count -text {text} -order {order} -unk -lm {name}.lm {smoothing}')

# Return the perplexity SRILM reports for a text file
def ppl(lm, text, extra=''):
    out = sh(f'./ngram -lm {lm} -unk -ppl {text} {extra}')
    return float(re.search(r'ppl= (\S+)', out).group(1))

# Read the <unk> probability back from the .lm file (stored as log10)
def unk_prob(lm):
    for line in open(lm):
        parts = line.rstrip('\n').split('\t')
        if len(parts) > 1 and parts[1] == '<unk>':
            return 10 ** float(parts[0])
    return 0.0

## Exercise 1: unsmoothed bigram model and the -inf problem

In [ ]:
# Exercise 1: bigram model with no smoothing (-addsmooth 0) and <unk> handling
!./ngram-count -text browndata/brown-train.txt -order 2 -write brown-train.count -unk -lm brown-train.lm -addsmooth 0

In [ ]:
# Test perplexity of the unsmoothed model
!./ngram -lm brown-train.lm -ppl browndata/brown-test.txt

In [ ]:
# Per-word log probabilities for the test set (first lines only)
!./ngram -lm brown-train.lm -ppl browndata/brown-test.txt -debug 2 | head -n 12

In [ ]:
# Count the words that got a probability of -inf (the closing bracket avoids
# matching words that merely contain '-inf')
!./ngram -lm brown-train.lm -ppl browndata/brown-test.txt -debug 2 | grep -c -- "-inf \]"

In [ ]:
# Probability SRILM gave to the <unk> token
!grep "<unk>" brown-train.lm | head -n 2

## Exercise 2: other smoothing methods (bigram)

In [ ]:
# Exercise 2: train a bigram model for each smoothing method and compare
# test perplexity and the probability mass given to <unk>
# SRILM options for each smoothing method
methods = {
    'No smoothing': '-addsmooth 0',
    'Add one': '-addsmooth 1',
    'Good-Turing': '',
    'Kneser-Ney': '-kndiscount -interpolate',
    'Witten-Bell (interpolated)': '-wbdiscount -interpolate',
    'Witten-Bell (plain)': '-wbdiscount',
}

rows = []
for name, option in methods.items():
    train('bi', 2, option)
    rows.append([name, ppl('bi.lm', test_file), unk_prob('bi.lm')])

bigram_table = pd.DataFrame(rows, columns=['Smoothing', 'Test perplexity', 'P(unk)'])
print(bigram_table.to_string(index=False))

## Exercise 3: trigram model

In [ ]:
# Exercise 3: same comparison for trigram models
rows = []
for name, option in methods.items():
    train('tri', 3, option)
    rows.append([name, ppl('tri.lm', test_file), unk_prob('tri.lm')])

trigram_table = pd.DataFrame(rows, columns=['Smoothing', 'Test perplexity', 'P(unk)'])
print(trigram_table.to_string(index=False))

In [ ]:
# Look at the per-word probabilities for one test sentence, bigram vs trigram
train('bi', 2, '-wbdiscount -interpolate')
train('tri', 3, '-wbdiscount -interpolate')
!head -n 1 browndata/brown-test.txt > one_sentence.txt
!./ngram -lm bi.lm -unk -ppl one_sentence.txt -debug 2 | head -n 12
!./ngram -lm tri.lm -unk -ppl one_sentence.txt -debug 2 | head -n 12

## Exercise 4: removing punctuation and stop words

In [ ]:
# Exercise 4: strip punctuation and/or stop words from train and test,
# then retrain and compare perplexity
# Write a cleaned copy of a file and return how many words it has
def clean(src, dst, drop_punct, drop_stop):
    count = 0
    with open(src) as f, open(dst, 'w') as g:
        for line in f:
            words = line.split()
            if drop_punct:
                words = [w for w in words if any(c.isalnum() for c in w)]
            if drop_stop:
                words = [w for w in words if w.lower() not in ENGLISH_STOP_WORDS]
            if words:
                g.write(' '.join(words) + '\n')
                count += len(words)
    return count

# (drop punctuation, drop stop words) for each setting
settings = {
    'Original': (False, False),
    'No punctuation': (True, False),
    'No stop words': (False, True),
    'Both removed': (True, True),
}

rows = []
for name, (p, s) in settings.items():
    clean(train_file, 'train_clean.txt', p, s)
    words = clean(test_file, 'test_clean.txt', p, s)
    row = [name, words]
    for order in (2, 3):
        train('clean', order, '-wbdiscount -interpolate', 'train_clean.txt')
        row.append(ppl('clean.lm', 'test_clean.txt'))
    rows.append(row)

prep_table = pd.DataFrame(rows, columns=['Preprocessing', 'Test words', 'Bigram', 'Trigram'])
print(prep_table.to_string(index=False))

## Exercise 5: interpolated unigram and bigram model

In [ ]:
# Exercise 5: grid search over the add-alpha value and the interpolation weight
# (lambda), scoring on the dev set
alphas = [0.00001, 0.0001, 0.0005, 0.001, 0.01, 0.1, 1]
lambdas = [i / 20 for i in range(1, 20)]

results = []
for a in alphas:
    train('uni', 1, f'-addsmooth {a}')
    train('bi', 2, f'-addsmooth {a}')
    for lam in lambdas:
        score = ppl('bi.lm', dev_file, f'-mix-lm uni.lm -lambda {lam}')
        results.append((a, lam, score))

grid = pd.DataFrame(results, columns=['addsmooth', 'lambda', 'dev_ppl'])
print(grid.pivot(index='lambda', columns='addsmooth', values='dev_ppl').round(0).to_string())

In [ ]:
# Pick the combination with the lowest dev perplexity
best = grid.loc[grid['dev_ppl'].idxmin()]
best_a, best_lam = best['addsmooth'], best['lambda']
print('best addsmooth:', best_a)
print('best lambda:', best_lam)
print('dev perplexity:', round(best['dev_ppl'], 2))

In [ ]:
# Retrain with the best add-alpha value and score once on the test set
train('uni', 1, f'-addsmooth {best_a}')
train('bi', 2, f'-addsmooth {best_a}')
print('test perplexity:', ppl('bi.lm', test_file, f'-mix-lm uni.lm -lambda {best_lam}'))